In [ ]:
from pyspark.sql.functions import *
from pyspark.sql import functions as F
import re
from pyspark.sql.window import Window
from pyspark.sql.functions import upper

In [ ]:
sfOptions_prod = {
  "sfURL" : "<your_company>.snowflakecomputing.com",
  "sfUser" : dbutils.secrets.get(scope="snowflake-secrets", key="snowflake-user"),
  "sfPassword" : dbutils.secrets.get(scope="snowflake-secrets", key="snowflake-password"),
  "sfDatabase" : "<database_name>",
  "sfSchema" : "<schema_name>",
  "sfWarehouse" : "<warehouse_name>",
  "truncate_table" : "ON",
  "usestagingtable" : "OFF"
}

# Snowflake source name for Spark
SNOWFLAKE_SOURCE_NAME = "net.snowflake.spark.snowflake"

# Read data from Snowflake for different entities with filtering

# Load products data from Snowflake
data_df = spark.read.format(SNOWFLAKE_SOURCE_NAME).options(**sfOptions_prod).option("dbtable", "sellout_deliveries_table").load()


In [ ]:
display(data_df)

In [ ]:
data_df.createOrReplaceTempView("sellout_all_deliveries")

In [ ]:
result_df = spark.sql("""
                      
-- =============================
-- DAIRY
-- =============================
SELECT
    'DAIRY_TOP_LEVEL' AS BUSINESS_SCOPE,
    MDM_CATEGORY_DSC,
    NULL AS MDM_SUB_CATEGORY_DSC,
    DELIVERY_PERIOD,
    PERIOD_TAG_DSC,
    FACT_TAG_DSC,
    COUNTRY_DSC,
    INCLUDE_STATUS,
    SUM(VALUE_NBR) AS VALUE_NBR
FROM sellout_all_deliveries
WHERE MDM_CATEGORY_DSC = 'DAIRY YOGHURT & DESSERTS'
  AND DELIVERY_PERIOD = 'P202513'
  AND PERIOD_TAG_DSC LIKE 'P%'
  AND FACT_TAG_DSC IN ('VOL','VAL')
  AND COUNTRY_DSC IN (
        'USA','BRAZIL','FRANCE','GERMANY','ITALY','POLAND','SPAIN',
        'UNITED KINGDOM','ALGERIA','EGYPT','TURKIYE','CANADA',
        'MEXICO','JAPAN','MOROCCO','SOUTH AFRICA'
  )
  AND INCLUDE_STATUS = 'INCLUDED'
  AND GL_LEVEL_DSC IN ('BOTH','VAL VOL ADDITIVE')
GROUP BY
    MDM_CATEGORY_DSC, DELIVERY_PERIOD, PERIOD_TAG_DSC,
    FACT_TAG_DSC, COUNTRY_DSC, INCLUDE_STATUS

UNION ALL

-- =============================
-- PLANT BASED
-- =============================
SELECT
    'PLANT_BASED',
    MDM_CATEGORY_DSC,
    NULL,
    DELIVERY_PERIOD,
    PERIOD_TAG_DSC,
    FACT_TAG_DSC,
    COUNTRY_DSC,
    INCLUDE_STATUS,
    SUM(VALUE_NBR)
FROM sellout_all_deliveries
WHERE MDM_CATEGORY_DSC IN (
        'PLANT BASED DRINKS',
        'PLANT BASED YOGHURT & DESSERTS'
)
  AND DELIVERY_PERIOD = 'P202513'
  AND PERIOD_TAG_DSC LIKE 'P%'
  AND FACT_TAG_DSC IN ('VOL','VAL')
  AND COUNTRY_DSC IN (
        'USA','FRANCE','GERMANY','ITALY','NETHERLANDS',
        'POLAND','SPAIN','UNITED KINGDOM',
        'CANADA','MEXICO','SWEDEN'
  )
  AND INCLUDE_STATUS = 'INCLUDED'
  AND GL_LEVEL_DSC IN ('BOTH','VAL VOL ADDITIVE')
GROUP BY
    MDM_CATEGORY_DSC, DELIVERY_PERIOD, PERIOD_TAG_DSC,
    FACT_TAG_DSC, COUNTRY_DSC, INCLUDE_STATUS

UNION ALL

-- =============================
-- WATERS
-- =============================
SELECT
    'WATERS',
    NULL,
    MDM_SUB_CATEGORY_DSC,
    DELIVERY_PERIOD,
    PERIOD_TAG_DSC,
    FACT_TAG_DSC,
    COUNTRY_DSC,
    INCLUDE_STATUS,
    SUM(VALUE_NBR)
FROM sellout_all_deliveries
WHERE MDM_SUB_CATEGORY_DSC IN (
        'PLAIN WATER JUG (HOD)',
        'PLAIN WATER STILL',
        'PLAIN WATER SPARKLING',
        'FLAVORED WATER STILL',
        'FLAVORED WATER SPARKLING'
)
  AND DELIVERY_PERIOD = 'P202513'
  AND PERIOD_TAG_DSC LIKE 'P%'
  AND FACT_TAG_DSC IN ('VOL','VAL')
  AND COUNTRY_DSC IN (
        'USA','FRANCE','GERMANY','POLAND','SPAIN',
        'UNITED KINGDOM','CHINA','INDONESIA',
        'TURKIYE','MEXICO','URUGUAY'
  )
  AND INCLUDE_STATUS = 'INCLUDED'
  AND GL_LEVEL_DSC IN ('BOTH','VAL VOL ADDITIVE')
GROUP BY
    MDM_SUB_CATEGORY_DSC, DELIVERY_PERIOD,
    PERIOD_TAG_DSC, FACT_TAG_DSC,
    COUNTRY_DSC, INCLUDE_STATUS

UNION ALL

-- =============================
-- WATERS CHINA
-- =============================
SELECT
    'WATERS',
    MDM_CATEGORY_DSC,
    NULL,
    DELIVERY_PERIOD,
    PERIOD_TAG_DSC,
    FACT_TAG_DSC,
    COUNTRY_DSC,
    INCLUDE_STATUS,
    SUM(VALUE_NBR)
FROM sellout_all_deliveries
WHERE MDM_CATEGORY_DSC IN ('OTHER DRINKS','FUNCTIONAL DRINKS')
  AND COUNTRY_DSC = 'CHINA'
  AND DELIVERY_PERIOD = 'P202513'
  AND PERIOD_TAG_DSC LIKE 'P%'
  AND FACT_TAG_DSC IN ('VOL','VAL')
  AND INCLUDE_STATUS = 'INCLUDED'
  AND GL_LEVEL_DSC IN ('BOTH','VAL VOL ADDITIVE')
GROUP BY
    MDM_CATEGORY_DSC, DELIVERY_PERIOD,
    PERIOD_TAG_DSC, FACT_TAG_DSC,
    COUNTRY_DSC, INCLUDE_STATUS

UNION ALL

-- =============================
-- SN ELN
-- =============================
SELECT
    'SN_ELN_PEADS',
    MDM_CATEGORY_DSC,
    MDM_SUB_CATEGORY_DSC,
    DELIVERY_PERIOD,
    PERIOD_TAG_DSC,
    FACT_TAG_DSC,
    COUNTRY_DSC,
    INCLUDE_STATUS,
    SUM(VALUE_NBR)
FROM sellout_all_deliveries
WHERE MDM_SUB_CATEGORY_DSC IN ('IF','FO','YCF 3+','YCF 1-3','GI')
  AND DELIVERY_PERIOD = 'P202513'
  AND COUNTRY_DSC IN ('USA','BRAZIL','AUSTRIA','BELGIUM','CZECH REPUBLIC','FRANCE','GERMANY','HUNGARY','IRELAND','ITALY','NETHERLANDS','POLAND','PORTUGAL','ROMANIA','SPAIN',
'SWITZERLAND','UNITED KINGDOM','AUSTRALIA','CHINA','HONGKONG','ALGERIA','EGYPT','INDIA','INDONESIA','THAILAND','MALAYSIA','SAUDI ARABIA','TURKIYE','UAE','ARGENTINA')
  AND PERIOD_TAG_DSC LIKE 'P%'
  AND FACT_TAG_DSC IN ('VOL','VAL')
  AND INCLUDE_STATUS = 'INCLUDED'
  AND GL_LEVEL_DSC IN ('BOTH','VAL VOL ADDITIVE')
GROUP BY
    MDM_CATEGORY_DSC, MDM_SUB_CATEGORY_DSC,
    DELIVERY_PERIOD, PERIOD_TAG_DSC,
    FACT_TAG_DSC, COUNTRY_DSC, INCLUDE_STATUS

UNION ALL

-- =============================
-- SN PEADS - ALLERGY
-- =============================
SELECT
    'SN_PEADS',
    MDM_CATEGORY_DSC,
    MDM_SUB_CATEGORY_DSC,
    DELIVERY_PERIOD,
    PERIOD_TAG_DSC,
    FACT_TAG_DSC,
    COUNTRY_DSC,
    INCLUDE_STATUS,
    SUM(VALUE_NBR)
FROM sellout_all_deliveries
WHERE MDM_SUB_CATEGORY_DSC = 'ALLERGY'
  AND DELIVERY_PERIOD = 'P202513'
  AND COUNTRY_DSC IN ('USA','BRAZIL','FRANCE','GERMANY','NETHERLANDS','POLAND','SPAIN','UNITED KINGDOM','AUSTRALIA','CHINA','ALGERIA','INDONESIA','THAILAND','MALAYSIA','SAUDI ARABIA','TURKIYE','UAE')
  AND PERIOD_TAG_DSC LIKE 'P%'
  AND FACT_TAG_DSC IN ('VOL','VAL')
  AND INCLUDE_STATUS = 'INCLUDED'
  AND GL_LEVEL_DSC IN ('BOTH','VAL VOL ADDITIVE')
GROUP BY
    MDM_CATEGORY_DSC, MDM_SUB_CATEGORY_DSC,
    DELIVERY_PERIOD, PERIOD_TAG_DSC,
    FACT_TAG_DSC, COUNTRY_DSC, INCLUDE_STATUS

UNION ALL

-- =============================
-- SN PEADS - CHALLENGED GROWTH
-- =============================
SELECT
    'SN_PEADS',
    MDM_CATEGORY_DSC,
    MDM_SUB_CATEGORY_DSC,
    DELIVERY_PERIOD,
    PERIOD_TAG_DSC,
    FACT_TAG_DSC,
    COUNTRY_DSC,
    INCLUDE_STATUS,
    SUM(VALUE_NBR)
FROM sellout_all_deliveries
WHERE MDM_SUB_CATEGORY_DSC = 'CHALLENGED GROWTH'
  AND DELIVERY_PERIOD = 'P202513'
  AND COUNTRY_DSC IN ('BRAZIL','GERMANY','NETHERLANDS','POLAND','SPAIN','UNITED KINGDOM','AUSTRALIA','CHINA','INDONESIA','THAILAND','MALAYSIA','SAUDI ARABIA','TURKIYE','UAE')
  AND PERIOD_TAG_DSC LIKE 'P%'
  AND FACT_TAG_DSC IN ('VOL','VAL')
  AND INCLUDE_STATUS = 'INCLUDED'
  AND GL_LEVEL_DSC IN ('BOTH','VAL VOL ADDITIVE')
GROUP BY
    MDM_CATEGORY_DSC, MDM_SUB_CATEGORY_DSC,
    DELIVERY_PERIOD, PERIOD_TAG_DSC,
    FACT_TAG_DSC, COUNTRY_DSC, INCLUDE_STATUS
;
""")

In [ ]:
# 2. Materialize in Databricks 
result_df = result_df.persist()

# 3. Force the execution (only once)
print(f"Total rows: {result_df.count()}")

In [ ]:
display(result_df)

Saving as Delta Table


In [ ]:
# Save as a Delta Table 
result_df.write.format("delta").mode("overwrite").saveAsTable("seasonality_base3")
print("Table correctly saved")

In [ ]:
seasonality_df = spark.table("seasonality_base3")
print(f"Rows: {seasonality_df.count()}")

DataFrame ready to be used

In [ ]:
display(seasonality_df)

In [ ]:
import pandas as pd

In [ ]:
df = seasonality_df.toPandas()

EXPLORATORY DATA ANALYSIS

In [ ]:
print("Shape:", df.shape)
print("\nColumnas:", df.columns.tolist())
print("\n Data Type:")
print(df.dtypes)
print("\n Complete info:")
df.info()

In [ ]:
print("Null values per columns:")
print(df.isnull().sum())
print("\n Percentage of nulls:")
print((df.isnull().sum() / len(df) * 100).round(2))

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    unique_vals = df[col].dropna().unique()
    print(f"\n--- {col} ({len(unique_vals)} unique values) ---")
    print(sorted(unique_vals))

Columns Distribution

In [ ]:
# How many entries per business_scope
print("=== BUSINESS_SCOPE ===")
print(df['BUSINESS_SCOPE'].value_counts())

# How many entries per country
print("\n=== COUNTRY_DSC ===")
print(df['COUNTRY_DSC'].value_counts())

# VOL vs VAL
print("\n=== FACT_TAG_DSC ===")
print(df['FACT_TAG_DSC'].value_counts())

# Available periods
print("\n=== PERIOD_TAG_DSC ===")
print(df['PERIOD_TAG_DSC'].value_counts().sort_index())

In [ ]:
print("=== VALUE_NBR ===")
print(df['VALUE_NBR'].describe())
print(f"\nNull values: {df['VALUE_NBR'].isnull().sum()}")
print(f"Values 0: {(df['VALUE_NBR'] == 0).sum()}")
print(f"Negative values: {(df['VALUE_NBR'] < 0).sum()}")

Creating dfs per business scope

In [ ]:
dfs = {scope: df[df['BUSINESS_SCOPE'] == scope].copy() 
       for scope in df['BUSINESS_SCOPE'].unique()}

# dfs per segment
df_dairy      = dfs['DAIRY_TOP_LEVEL']
df_plant      = dfs['PLANT_BASED']
df_waters     = dfs['WATERS']
df_sn_eln     = dfs['SN_ELN_PEADS']
df_sn_peads   = dfs['SN_PEADS']

for name, subset in dfs.items():
    print(f"{name}: {len(subset)} rows | {subset['COUNTRY_DSC'].nunique()} countries | {subset['PERIOD_TAG_DSC'].nunique()} periods")

In [ ]:
for name in dfs:
    dfs[name] = dfs[name].sort_values(
        ['COUNTRY_DSC', 'MDM_CATEGORY_DSC', 'MDM_SUB_CATEGORY_DSC', 
         'FACT_TAG_DSC', 'PERIOD_TAG_DSC']
    ).reset_index(drop=True)

print("Subtables created correctly.")

Coefficient Calculation

In [ ]:
# Excluiding P202213
df_base = df[df['PERIOD_TAG_DSC'] != 'P202213'].copy()

# Extract year and Period
df_base['YEAR'] = df_base['PERIOD_TAG_DSC'].str[1:5].astype(int)
df_base['PERIOD_NUM'] = df_base['PERIOD_TAG_DSC'].str[5:].astype(int)

# Sorting the table
df_base = df_base.sort_values(
    ['BUSINESS_SCOPE', 'COUNTRY_DSC', 'MDM_CATEGORY_DSC',
     'MDM_SUB_CATEGORY_DSC', 'FACT_TAG_DSC', 'YEAR', 'PERIOD_NUM']
).reset_index(drop=True)

print(df_base.shape)
df_base.head(20)

In [ ]:
df_base["BUSINESS_SCOPE"].unique()

Coeffients function

In [ ]:
def calculate_coefficients(df_prepared, scope_name):
    """
    Receives an already prepared df, with GRAIN_ID defined
    Returns the coefficients table for that scope
    """
    df = df_prepared.copy()
    df = df.sort_values(['GRAIN_ID', 'FACT_TAG_DSC', 'YEAR', 'PERIOD_NUM']).reset_index(drop=True)

    # ── Detect full years per GRAIN_ID ─────────────────────────────────
    OPTIONAL_PERIODS = {8}
    df_no_p8 = df[~df['PERIOD_NUM'].isin(OPTIONAL_PERIODS)]

    latest_year_per_grain = (
        df_no_p8.groupby('GRAIN_ID')['YEAR']
        .max()
        .reset_index(name='LATEST_YEAR')
    )

    df_reference = df_no_p8.merge(latest_year_per_grain, on='GRAIN_ID', how='left')
    df_reference = df_reference[df_reference['YEAR'] == df_reference['LATEST_YEAR']]

    standard_periods = (
        df_reference.groupby('GRAIN_ID')['PERIOD_NUM']
        .apply(set)
        .reset_index(name='EXPECTED_PERIODS')
    )

    periods_per_year = (
        df_no_p8.groupby(['GRAIN_ID', 'YEAR'])['PERIOD_NUM']
        .apply(set)
        .reset_index(name='ACTUAL_PERIODS')
    )

    periods_per_year = periods_per_year.merge(standard_periods, on='GRAIN_ID', how='left')
    periods_per_year['YEAR_VALID'] = (
        periods_per_year['ACTUAL_PERIODS'] == periods_per_year['EXPECTED_PERIODS']
    )

    valid_years = periods_per_year[periods_per_year['YEAR_VALID']][['GRAIN_ID', 'YEAR']]
    df = df.merge(valid_years, on=['GRAIN_ID', 'YEAR'], how='inner')

    # Step 1: L3M crossing years
    df = df.sort_values(['GRAIN_ID', 'FACT_TAG_DSC', 'YEAR', 'PERIOD_NUM']).reset_index(drop=True)
    df['L3M'] = (
        df.groupby(['GRAIN_ID', 'FACT_TAG_DSC'])['VALUE_NBR']
        .transform(lambda s: s.rolling(window=3, min_periods=3).sum())
    )
    
    # Step 2: Keep only periods with a full window (P3 >)
    # Whenever needed P1 and P2 if new data is available, apply this instead in step 2:
    # df_l3m = df[df['L3M'].notna()].copy()

    df_l3m = df[df['PERIOD_NUM'] >= 3].copy()

    # Step 3: Calculate L3M per general period
    df_consolidated = (
        df_l3m.groupby(['GRAIN_ID', 'FACT_TAG_DSC', 'PERIOD_NUM'], as_index=False)
        .agg(L3M_TOTAL=('L3M', 'sum'))
    )

    # Step 4: Historic total for full years
    df_total = (
        df.groupby(['GRAIN_ID', 'FACT_TAG_DSC'], as_index=False)
        .agg(TOTAL_HISTORICO=('VALUE_NBR', 'sum'))
    )

    # Step 5: Calculate coefficients
    df_coef = df_consolidated.merge(df_total, on=['GRAIN_ID', 'FACT_TAG_DSC'], how='left')
    df_coef['COEFFICIENT'] = df_coef['L3M_TOTAL'] / df_coef['TOTAL_HISTORICO']
    df_coef['BUSINESS_SCOPE'] = scope_name

    return df_coef[['BUSINESS_SCOPE', 'GRAIN_ID', 'FACT_TAG_DSC',
                     'PERIOD_NUM', 'L3M_TOTAL', 'TOTAL_HISTORICO', 'COEFFICIENT']]

##Dairy

In [ ]:
def prepare_dairy(df_base):
    df = df_base[df_base['BUSINESS_SCOPE'] == 'DAIRY_TOP_LEVEL'].copy()
    df['GRAIN_ID'] = df['COUNTRY_DSC'] + '|DAIRY|' + df['MDM_CATEGORY_DSC']
    return df

df_dairy_prep = prepare_dairy(df_base)
df_coef_dairy = calculate_coefficients(df_dairy_prep, 'DAIRY_TOP_LEVEL')

print(f"Dairy: {len(df_coef_dairy)} coefficients | {df_coef_dairy['GRAIN_ID'].nunique()} grains")
df_coef_dairy.head(50)

###Plant Based

In [ ]:
def prepare_plant(df_base):
    df = df_base[df_base['BUSINESS_SCOPE'] == 'PLANT_BASED'].copy()
    df['GRAIN_ID'] = df['COUNTRY_DSC'] + '|PLANT_BASED|' + df['MDM_CATEGORY_DSC']
    return df

df_plant_prep = prepare_plant(df_base)
df_coef_plant = calculate_coefficients(df_plant_prep, 'PLANT_BASED')

print(f"Plant Based: {len(df_coef_plant)} coefficients | {df_coef_plant['GRAIN_ID'].nunique()} grains")
df_coef_plant.head(10)

Check df for dairy

In [ ]:
display(df_coef_dairy)

###Waters

Checking for china's special subcategory

In [ ]:
waters_china_cats = df_base[
    (df_base['BUSINESS_SCOPE'] == 'WATERS') & 
    (df_base['COUNTRY_DSC'] == 'CHINA') &
    (df_base['MDM_CATEGORY_DSC'].isin(['OTHER DRINKS', 'FUNCTIONAL DRINKS']))
][['MDM_CATEGORY_DSC', 'MDM_SUB_CATEGORY_DSC']].drop_duplicates().sort_values(['MDM_CATEGORY_DSC', 'MDM_SUB_CATEGORY_DSC'])

print("OTHER DRINKS and FUNCTIONAL DRINKS in China ===")
print(waters_china_cats.to_string(index=False))

# All water grains in China
waters_china_all = df_base[
    (df_base['BUSINESS_SCOPE'] == 'WATERS') & 
    (df_base['COUNTRY_DSC'] == 'CHINA')
][['MDM_CATEGORY_DSC', 'MDM_SUB_CATEGORY_DSC']].drop_duplicates().sort_values(['MDM_CATEGORY_DSC', 'MDM_SUB_CATEGORY_DSC'])

print("\n=== Water grains in China ===")
print(waters_china_all.to_string(index=False))

In [ ]:
def prepare_waters(df_base):
    df = df_base[df_base['BUSINESS_SCOPE'] == 'WATERS'].copy()

    # Creating CHINA SUPER DRINKS OTHER/FUNCTIONAL del resto
    china_super = df[
        (df['COUNTRY_DSC'] == 'CHINA') &
        (df['MDM_CATEGORY_DSC'].isin(['OTHER DRINKS', 'FUNCTIONAL DRINKS']))
    ].copy()
    
    resto = df[
        ~((df['COUNTRY_DSC'] == 'CHINA') &
          (df['MDM_CATEGORY_DSC'].isin(['OTHER DRINKS', 'FUNCTIONAL DRINKS'])))
    ].copy()
    
    if len(china_super) > 0:
        china_fusionado = china_super.groupby(
            ['BUSINESS_SCOPE', 'YEAR','PERIOD_NUM', 'COUNTRY_DSC', 'PERIOD_TAG_DSC', 'FACT_TAG_DSC'],
            as_index=False
        )['VALUE_NBR'].sum()
        
        # Agregate the subcategory SUPER DRINKS
        china_fusionado['MDM_CATEGORY_DSC'] = None
        china_fusionado['MDM_SUB_CATEGORY_DSC'] = 'SUPER DRINKS'
        china_fusionado['BUSINESS_SCOPE'] = 'WATERS'
        china_fusionado['INCLUDE_STATUS'] = 'INCLUDED'
        china_fusionado['DELIVERY_PERIOD'] = df['DELIVERY_PERIOD'].iloc[0] if 'DELIVERY_PERIOD' in df.columns else None
        
        # Concatenate
        df_final = pd.concat([resto, china_fusionado], ignore_index=True)
    else:
        df_final = resto
    
    # GRAIN_ID definition
    df_final['GRAIN_ID'] = (
        df_final['COUNTRY_DSC'] + '|WATERS|' + df_final['MDM_SUB_CATEGORY_DSC']
    )
    
    return df_final

In [ ]:
# 1. Prepping Waters
df_waters_prep = prepare_waters(df_base)

print(f"Waters prepared: {len(df_waters_prep)} rows")
print(f"Unique grains: {df_waters_prep['GRAIN_ID'].nunique()}")
print(f"Subcategories: {sorted(df_waters_prep['MDM_SUB_CATEGORY_DSC'].unique())}")

# Verify SUPER DRINKS
if 'SUPER DRINKS' in df_waters_prep['MDM_SUB_CATEGORY_DSC'].values:
    super_drinks = df_waters_prep[
        df_waters_prep['MDM_SUB_CATEGORY_DSC'] == 'SUPER DRINKS'
    ]['GRAIN_ID'].unique()
    print(f"\nChina SUPER DRINKS grains: {super_drinks}")



In [ ]:
# 2. Calculate coefficients
df_coef_waters = calculate_coefficients(df_waters_prep, 'WATERS')

print(f"\n=== Coefficients Waters ===")
print(f"Rows: {len(df_coef_waters)}")
print(f"Grains: {df_coef_waters['GRAIN_ID'].nunique()}")
print(f"Perios: {sorted(df_coef_waters['PERIOD_NUM'].unique())}")

# 3. See some coefficients
print("\n=== First coefficients ===")
display(df_coef_waters)

In [ ]:
print("=== Countries in waters ===")
print(df_waters_prep['COUNTRY_DSC'].value_counts().sort_index())

print(f"\nTotal countries: {df_waters_prep['COUNTRY_DSC'].nunique()}")

# Grains per country
print("\n=== Grains per country ===")
grains_per_country = df_waters_prep.groupby('COUNTRY_DSC')['GRAIN_ID'].nunique()
print(grains_per_country.sort_values(ascending=False))

In [ ]:
# Check how many years does CHINA SUPER DRINKS have
china_super = df_waters_prep[
    df_waters_prep['GRAIN_ID'] == 'CHINA|WATERS|SUPER DRINKS'
]

print("=== China SUPER DRINKS ===")
print(f"Rows: {len(china_super)}")
print(f"Avaliable years: {sorted(china_super['YEAR'].unique())}")
print(f"Number of years: {china_super['YEAR'].nunique()}")

# Distribution per year and fact tag
print("\n=== Per year and fact tag ===")
print(china_super.groupby(['YEAR', 'FACT_TAG_DSC']).size().unstack())

### SN_ELN_PEADS

####Structure

In [ ]:
# SN_ELN_PEADS structure
eln_full = df_base[df_base['BUSINESS_SCOPE'] == 'SN_ELN_PEADS'][
    ['COUNTRY_DSC', 'MDM_CATEGORY_DSC', 'MDM_SUB_CATEGORY_DSC']
].drop_duplicates().sort_values(['COUNTRY_DSC', 'MDM_CATEGORY_DSC', 'MDM_SUB_CATEGORY_DSC'])

print("=== SN_ELN_PEADS structures ===")
print(eln_full.to_string(index=False))

print(f"\nCountries: {df_base[df_base['BUSINESS_SCOPE'] == 'SN_ELN_PEADS']['COUNTRY_DSC'].nunique()}")
print(f"Subcategories: {df_base[df_base['BUSINESS_SCOPE'] == 'SN_ELN_PEADS']['MDM_SUB_CATEGORY_DSC'].unique()}")

# Check for Brazil specifically
print("\n=== Brazil SN_ELN_PEADS ===")
brazil_eln = df_base[
    (df_base['BUSINESS_SCOPE'] == 'SN_ELN_PEADS') & 
    (df_base['COUNTRY_DSC'] == 'BRAZIL')
][['MDM_SUB_CATEGORY_DSC', 'YEAR']].drop_duplicates().sort_values(['MDM_SUB_CATEGORY_DSC', 'YEAR'])
print(brazil_eln.to_string(index=False))

In [ ]:
def prepare_sn_eln_peads(df_base):
    """
    Prepare SN_ELN_PEADS:
    - Granularity: MDM_SUB_CATEGORY_DSC
    - Special case: YCF 1-3 + YCF 3+ → YCF
    """
    df = df_base[df_base['BUSINESS_SCOPE'] == 'SN_ELN_PEADS'].copy()
    
    # Separate YCF 1-3 y YCF 3+ from the whole
    ycf_1_3 = df[df['MDM_SUB_CATEGORY_DSC'] == 'YCF 1-3'].copy()
    ycf_3_plus = df[df['MDM_SUB_CATEGORY_DSC'] == 'YCF 3+'].copy()
    resto = df[~df['MDM_SUB_CATEGORY_DSC'].isin(['YCF 1-3', 'YCF 3+'])].copy()
    
    # Sum YCF 1-3 + YCF 3+ → YCF
    if len(ycf_1_3) > 0 and len(ycf_3_plus) > 0:
        ycf_3_plus['MDM_SUB_CATEGORY_DSC'] = 'YCF'
        ycf_1_3['MDM_SUB_CATEGORY_DSC'] = 'YCF'
        
        # Sum both categories per grain + year + period + fact
        ycf_fusionado = pd.concat([ycf_1_3, ycf_3_plus], ignore_index=True)
        
        #   Groupby to sum the values
        ycf_fusionado = ycf_fusionado.groupby(
            ['BUSINESS_SCOPE', 'COUNTRY_DSC', 'MDM_CATEGORY_DSC', 
             'MDM_SUB_CATEGORY_DSC', 'PERIOD_TAG_DSC', 'FACT_TAG_DSC',
             'YEAR', 'PERIOD_NUM', 'INCLUDE_STATUS', 'DELIVERY_PERIOD'],
            as_index=False
        )['VALUE_NBR'].sum()
        
        # Concatenate
        df_final = pd.concat([resto, ycf_fusionado], ignore_index=True)
    else:
        if len(ycf_1_3) > 0:
            ycf_1_3['MDM_SUB_CATEGORY_DSC'] = 'YCF'
            df_final = pd.concat([resto, ycf_1_3], ignore_index=True)
        elif len(ycf_3_plus) > 0:
            ycf_3_plus['MDM_SUB_CATEGORY_DSC'] = 'YCF'
            df_final = pd.concat([resto, ycf_3_plus], ignore_index=True)
        else:
            df_final = resto
    
    # Create GRAIN_ID
    df_final['GRAIN_ID'] = (
        df_final['COUNTRY_DSC'] + '|SN_ELN_PEADS|' + 
        df_final['MDM_SUB_CATEGORY_DSC']
    )
    
    return df_final

df_eln_prep = prepare_sn_eln_peads(df_base)

print(f"SN_ELN_PEADS prepped: {len(df_eln_prep)} filas")
print(f"Unique grains: {df_eln_prep['GRAIN_ID'].nunique()}")
print(f"Subcategories: {sorted(df_eln_prep['MDM_SUB_CATEGORY_DSC'].unique())}")

print("\n=== Verify fusioned YCF ===")
ycf_check = df_eln_prep[df_eln_prep['MDM_SUB_CATEGORY_DSC'] == 'YCF']
print(f"Grains YCF: {ycf_check['GRAIN_ID'].nunique()}")
print(f"Countries in YCF: {ycf_check['COUNTRY_DSC'].nunique()}")

# See Brazil YCF
print("\n=== Brazil YCF ===")
brazil_ycf = ycf_check[ycf_check['COUNTRY_DSC'] == 'BRAZIL']
print(f"Years: {sorted(brazil_ycf['YEAR'].dropna().unique())}")
print(f"Rows: {len(brazil_ycf)}")

# Calculate coefficients
df_coef_eln = calculate_coefficients(df_eln_prep, 'SN_ELN_PEADS')

print(f"\n=== Coefficients SN_ELN_PEADS ===")
print(f"Rows: {len(df_coef_eln)}")
print(f"Grains: {df_coef_eln['GRAIN_ID'].nunique()}")
print(f"Periods: {sorted(df_coef_eln['PERIOD_NUM'].unique())}")

display(df_coef_eln)

###SN_PEADS

In [ ]:
#Structure SN_PEADS 
peads_full = df_base[df_base['BUSINESS_SCOPE'] == 'SN_PEADS'][
    ['COUNTRY_DSC', 'MDM_CATEGORY_DSC', 'MDM_SUB_CATEGORY_DSC']
].drop_duplicates().sort_values(['COUNTRY_DSC', 'MDM_CATEGORY_DSC', 'MDM_SUB_CATEGORY_DSC'])

print("=== SN_PEADS Structure ===")
print(peads_full.to_string(index=False))

print(f"\nCountries: {df_base[df_base['BUSINESS_SCOPE'] == 'SN_PEADS']['COUNTRY_DSC'].nunique()}")
print(f"Subcategories: {df_base[df_base['BUSINESS_SCOPE'] == 'SN_PEADS']['MDM_SUB_CATEGORY_DSC'].unique()}")

# See Brazil
print("\n=== Brazil SN_PEADS ===")
brazil_peads = df_base[
    (df_base['BUSINESS_SCOPE'] == 'SN_PEADS') & 
    (df_base['COUNTRY_DSC'] == 'BRAZIL')
][['MDM_SUB_CATEGORY_DSC', 'YEAR']].drop_duplicates().sort_values(['MDM_SUB_CATEGORY_DSC', 'YEAR'])
print(brazil_peads.to_string(index=False))

### Peads Preparation

In [ ]:

def prepare_sn_peads(df_base):
    """
    Prepares SN_PEADS:
    - Granularity: MDM_SUB_CATEGORY_DSC (ALLERGY, CHALLENGED GROWTH)
    - No special cases
    - Brazil CHALLENGED GROWTH has only 2 years 
    """
    df = df_base[df_base['BUSINESS_SCOPE'] == 'SN_PEADS'].copy()

    # Create GRAIN_ID
    df['GRAIN_ID'] = (
        df['COUNTRY_DSC'] + '|SN_PEADS|' + df['MDM_SUB_CATEGORY_DSC']
    )

    return df

# Testing preparation
df_peads_prep = prepare_sn_peads(df_base)

print(f"SN_PEADS prepared: {len(df_peads_prep)} filas")
print(f"Unique grains: {df_peads_prep['GRAIN_ID'].nunique()}")
print(f"Subcategories: {sorted(df_peads_prep['MDM_SUB_CATEGORY_DSC'].unique())}")


# Calculate coefficients
df_coef_peads = calculate_coefficients(df_peads_prep, 'SN_PEADS')

print(f"\n=== Coefficients SN_PEADS ===")
print(f"Rows: {len(df_coef_peads)}")
print(f"Grains: {df_coef_peads['GRAIN_ID'].nunique()}")
print(f"Periods: {sorted(df_coef_peads['PERIOD_NUM'].unique())}")

In [ ]:
display(df_coef_peads)

###Concatening the results

In [ ]:
df_coef_master = pd.concat([
    df_coef_dairy,
    df_coef_plant,
    df_coef_waters,
    df_coef_eln,
    df_coef_peads
], ignore_index=True)

print(f"Total rows: {len(df_coef_master)}")
print(f"Total grains: {df_coef_master['GRAIN_ID'].nunique()}")
print(df_coef_master['BUSINESS_SCOPE'].value_counts())

### Checking master table

In [ ]:
print("Rows:", len(df_coef_master))
print("Unique rows:", len(df_coef_master.drop_duplicates()))
print("Duplicate rows:", len(df_coef_master) - len(df_coef_master.drop_duplicates()))

key_cols = ["BUSINESS_SCOPE", "GRAIN_ID", "FACT_TAG_DSC", "PERIOD_NUM"]
dup_check = df_coef_master.groupby(key_cols).size().reset_index(name="count")
print(dup_check[dup_check["count"] > 1].head(20))

### Adjusting the output format to match PowerBI's

In [ ]:
# 1. Extract each column from GRAIN_ID
df_coef_master[['COUNTRY', 'SCOPE', 'CATEGORY']] = (
    df_coef_master['GRAIN_ID'].str.split('|', expand=True)
)

# 2. New DF in the expected format
df_output = df_coef_master[[
    'COUNTRY',
    'CATEGORY',
    'FACT_TAG_DSC',
    'PERIOD_NUM',
    'COEFFICIENT'
]].copy()

# 3. Renaming columns
df_output = df_output.rename(columns={
    'FACT_TAG_DSC': 'METRIC',
    'PERIOD_NUM': 'PERIOD',
    'COEFFICIENT': 'VALUE'
})

# 4. Sorting
df_output = (
    df_output
    .sort_values(['COUNTRY', 'CATEGORY', 'METRIC', 'PERIOD'])
    .reset_index(drop=True)
)

display(df_output)
print(df_output.columns.tolist())

### Output to excel

In [ ]:
# Export df_output to Excel
!pip install openpyxl
df_output.to_excel("./output/coefficients.xlsx", index=False)

print("Saved successfully")
